
# Multiperiod deterministic arbitrage detection

This Colab notebook implements the **multiperiod deterministic arbitrage detector** under the physical-settlement / passive-carry convention.

The notebook follows the multi-maturity construction of the paper:

- all positions are chosen at time zero;
- intermediate option exercises mechanically create stock-and-cash positions;
- those positions are carried passively to the common final horizon;
- the pathwise no-arbitrage test is reduced to finitely many linear inequalities by enumerating pre-final exercise patterns and checking final-maturity breakpoints plus tail slopes;
- bid-ask execution is handled directly: purchases use ask prices and sales use bid prices.

The main function is:

```python
solve_multiperiod_arbitrage_detection(...)
```

It returns a SciPy LP result and a model object. If the maximized margin objective is positive, the extracted portfolio is a deterministic multiperiod arbitrage certificate.



## Cell 1 - Install packages

Run this cell first in Colab. The implementation uses only standard scientific Python packages.


In [ ]:
%pip -q install numpy pandas scipy matplotlib


## Cell 2 - Full implementation

This cell defines the complete multiperiod arbitrage detector.

The code is intentionally verbose and heavily commented so that each mathematical object in the paper has a visible programming counterpart.


In [ ]:

# ============================================================
# Multiperiod deterministic arbitrage detector
# Physical settlement + passive carry, with bid-ask execution
# ============================================================
#
# This code implements the multiperiod extension of the deterministic
# arbitrage detector described in Section 8 of the paper:
#
#   "Deterministic Arbitrage, Localized Arbitrage Portfolios,
#    and Arbitrage-Consistent Projection"
#
# The financial convention is the important part:
#
# 1. All positions are chosen at time zero. There is NO discretionary
#    rebalancing at intermediate maturities.
#
# 2. If an option matures before the final horizon and finishes in the
#    money, physical settlement mechanically creates a stock-and-cash
#    position. That stock-and-cash position is then carried passively
#    to the final horizon.
#
# 3. Options with zero intrinsic value at maturity are assumed NOT to
#    be physically settled. Therefore:
#
#       call settles only when S_Tj > K,
#       put  settles only when S_Tj < K.
#
# 4. Bid-ask execution is handled by split nonnegative variables:
#
#       buy quantity  >= 0, executed at ask,
#       sell quantity >= 0, executed at bid.
#
#    The net position is buy - sell.
#
# 5. The no-arbitrage test is not imposed on a stochastic tree and does
#    not use probabilities. It is pathwise: terminal wealth must be
#    nonnegative for every possible path of the underlying.
#
# The key finite-dimensional reduction:
#
# - Each pre-final maturity generates finitely many possible exercise
#   patterns: open intervals between strikes plus singleton states at
#   strikes.
#
# - Conditional on one pre-final exercise pattern, the terminal wealth
#   is a one-dimensional piecewise-affine function of the final stock
#   price.
#
# - Therefore global nonnegativity is equivalent to checking the final
#   maturity breakpoints and the right-tail slope, pattern by pattern.
#
# The LP below searches for a zero-cost executable portfolio whose
# finite nonnegativity margins are positive. If the optimal value is
# positive, the returned positions form a deterministic multiperiod
# arbitrage under the physical-settlement convention.

import itertools
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from scipy.optimize import linprog

try:
    from IPython.display import display
except Exception:
    display = print


# ------------------------------------------------------------
# 1. Input schema and validation
# ------------------------------------------------------------

REQUIRED_BID_ASK_COLUMNS = [
    "T", "K",
    "call_bid", "call_ask",
    "put_bid", "put_ask",
]


def make_bid_ask_from_mid(
    quotes_mid: pd.DataFrame,
    call_col: str = "call",
    put_col: str = "put",
    spread: float = 0.0,
) -> pd.DataFrame:
    """Convert frictionless/mid quotes into a bid-ask quote table.

    Parameters
    ----------
    quotes_mid:
        DataFrame with at least columns T, K, call_col, put_col.
    call_col, put_col:
        Names of the columns containing one call price and one put price.
    spread:
        Absolute spread added symmetrically around the mid price.
        If spread=0, bid=ask=mid.

    Returns
    -------
    DataFrame with the bid-ask schema required by the detector.

    Notes
    -----
    The detector is written in bid-ask form. A frictionless market is the
    special case bid = ask.
    """
    df = quotes_mid.copy()
    half = 0.5 * float(spread)
    out = pd.DataFrame({
        "T": df["T"].astype(float),
        "K": df["K"].astype(float),
        "call_bid": df[call_col].astype(float) - half,
        "call_ask": df[call_col].astype(float) + half,
        "put_bid": df[put_col].astype(float) - half,
        "put_ask": df[put_col].astype(float) + half,
    })
    return out


def clean_multiperiod_bid_ask_quotes(quotes: pd.DataFrame) -> pd.DataFrame:
    """Validate and sort a multi-maturity bid-ask option table.

    Required columns:
        T, K, call_bid, call_ask, put_bid, put_ask

    Here T is time to maturity in years and K is strike.

    The function keeps the input format deliberately simple so that the
    notebook can be used directly in Colab with manually entered data,
    CSV uploads, or data frames built from an external data source.
    """
    missing = [c for c in REQUIRED_BID_ASK_COLUMNS if c not in quotes.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = quotes[REQUIRED_BID_ASK_COLUMNS].copy()
    for col in REQUIRED_BID_ASK_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="raise")

    if (df["T"] <= 0).any():
        raise ValueError("All maturities T must be strictly positive.")
    if (df["K"] <= 0).any():
        raise ValueError("All strikes K must be strictly positive.")
    if (df["call_bid"] > df["call_ask"]).any():
        raise ValueError("Found at least one call quote with call_bid > call_ask.")
    if (df["put_bid"] > df["put_ask"]).any():
        raise ValueError("Found at least one put quote with put_bid > put_ask.")
    if df.duplicated(["T", "K"]).any():
        dups = df[df.duplicated(["T", "K"], keep=False)]
        raise ValueError(f"Duplicate (T, K) rows found:\n{dups}")

    df = df.sort_values(["T", "K"]).reset_index(drop=True)
    return df


def build_market_book(
    quotes: pd.DataFrame,
    S0: float,
    r: float,
    stock_bid: Optional[float] = None,
    stock_ask: Optional[float] = None,
) -> Dict[str, Any]:
    """Build the internal market representation used by the LP.

    Parameters
    ----------
    quotes:
        Clean or raw bid-ask quote table with columns
        T, K, call_bid, call_ask, put_bid, put_ask.
    S0:
        Current stock price. Used as the frictionless stock price if
        stock_bid and stock_ask are not supplied.
    r:
        Continuously compounded risk-free rate.
    stock_bid, stock_ask:
        Optional stock bid and ask. If omitted, the stock is treated as
        frictionless with stock_bid = stock_ask = S0.

    Returns
    -------
    A dictionary containing per-maturity arrays and instrument records.
    """
    df = clean_multiperiod_bid_ask_quotes(quotes)

    S0 = float(S0)
    r = float(r)
    if S0 <= 0:
        raise ValueError("S0 must be strictly positive.")

    if stock_bid is None:
        stock_bid = S0
    if stock_ask is None:
        stock_ask = S0
    stock_bid = float(stock_bid)
    stock_ask = float(stock_ask)
    if stock_bid > stock_ask:
        raise ValueError("stock_bid cannot be greater than stock_ask.")

    maturities = sorted(df["T"].unique().astype(float))
    m = len(maturities)
    if m == 0:
        raise ValueError("At least one maturity is required.")

    strikes = []
    call_bid = []
    call_ask = []
    put_bid = []
    put_ask = []

    for T in maturities:
        g = df[df["T"] == T].sort_values("K")
        strikes.append(g["K"].to_numpy(float))
        call_bid.append(g["call_bid"].to_numpy(float))
        call_ask.append(g["call_ask"].to_numpy(float))
        put_bid.append(g["put_bid"].to_numpy(float))
        put_ask.append(g["put_ask"].to_numpy(float))

    option_records = []
    for j, T in enumerate(maturities):
        for i, K in enumerate(strikes[j]):
            option_records.append({
                "kind": "call",
                "j": j,
                "i": i,
                "T": float(T),
                "K": float(K),
                "bid": float(call_bid[j][i]),
                "ask": float(call_ask[j][i]),
                "name": f"C(T={T:g}, K={K:g})",
            })
            option_records.append({
                "kind": "put",
                "j": j,
                "i": i,
                "T": float(T),
                "K": float(K),
                "bid": float(put_bid[j][i]),
                "ask": float(put_ask[j][i]),
                "name": f"P(T={T:g}, K={K:g})",
            })

    return {
        "quotes": df,
        "S0": S0,
        "r": r,
        "stock_bid": stock_bid,
        "stock_ask": stock_ask,
        "maturities": maturities,
        "m": m,
        "T_final": maturities[-1],
        "strikes": strikes,
        "call_bid": call_bid,
        "call_ask": call_ask,
        "put_bid": put_bid,
        "put_ask": put_ask,
        "option_records": option_records,
    }


# ------------------------------------------------------------
# 2. Exercise-pattern enumeration
# ------------------------------------------------------------

def enumerate_exercise_patterns_for_strikes(
    strikes: np.ndarray,
    maturity_label: str = "",
) -> List[Dict[str, Any]]:
    """Enumerate all reachable call/put exercise patterns at one maturity.

    For strikes K_1 < ... < K_n, the relevant regions are:

        s < K_1,
        s = K_1,
        K_1 < s < K_2,
        s = K_2,
        ...
        s = K_n,
        s > K_n.

    This gives 2n + 1 patterns.

    Calls settle when s > K.
    Puts settle when s < K.
    At s = K, neither option with strike K is settled.
    """
    K = np.asarray(strikes, dtype=float)
    n = len(K)

    if n == 0:
        return [{
            "label": f"{maturity_label}: no strikes",
            "representative": np.nan,
            "call": np.zeros(0),
            "put": np.zeros(0),
        }]

    patterns = []

    # Region below the first strike. Since strikes are strictly positive,
    # representative s=0 belongs to this region.
    reps_and_labels = [(0.0, f"{maturity_label}: s < {K[0]:g}")]

    for i, k in enumerate(K):
        reps_and_labels.append((float(k), f"{maturity_label}: s = {k:g}"))
        if i < n - 1:
            mid = 0.5 * (K[i] + K[i + 1])
            reps_and_labels.append((float(mid), f"{maturity_label}: {K[i]:g} < s < {K[i+1]:g}"))

    # Region above the last strike.
    above = float(K[-1] + max(1.0, 0.10 * abs(K[-1])))
    reps_and_labels.append((above, f"{maturity_label}: s > {K[-1]:g}"))

    for rep, label in reps_and_labels:
        call_ind = (rep > K).astype(float)
        put_ind = (rep < K).astype(float)
        patterns.append({
            "label": label,
            "representative": rep,
            "call": call_ind,
            "put": put_ind,
        })

    return patterns


def enumerate_prefinal_patterns(book: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Enumerate all pre-final exercise-pattern combinations.

    If there are m maturities, only T_1,...,T_{m-1} are pattern variables.
    The final maturity is handled by the final stock price x directly.
    """
    m = book["m"]
    if m == 1:
        return [{
            "label": "no pre-final maturities",
            "components": [],
        }]

    patterns_by_maturity = []
    for j in range(m - 1):
        label = f"T{j+1}={book['maturities'][j]:g}"
        patterns_by_maturity.append(
            enumerate_exercise_patterns_for_strikes(book["strikes"][j], maturity_label=label)
        )

    all_patterns = []
    for combo in itertools.product(*patterns_by_maturity):
        all_patterns.append({
            "label": " | ".join(p["label"] for p in combo),
            "components": list(combo),
        })

    return all_patterns


def final_certificate_points(book: Dict[str, Any]) -> np.ndarray:
    """Return the final-maturity endpoint/breakpoint set {0, final strikes}."""
    final_strikes = np.asarray(book["strikes"][-1], dtype=float)
    points = np.unique(np.concatenate([np.array([0.0]), final_strikes]))
    return np.sort(points)


# ------------------------------------------------------------
# 3. Physical-settlement terminal payoff coefficients
# ------------------------------------------------------------

def option_terminal_coefficient(
    book: Dict[str, Any],
    option: Dict[str, Any],
    pattern: Dict[str, Any],
    x: float,
) -> float:
    """Coefficient of one net option position in terminal wealth.

    A "net option position" means:
        net = buy_quantity - sell_quantity.

    The LP uses split buy/sell variables, so this coefficient will be
    added to the buy variable and subtracted from the sell variable.

    For an option maturing before the final horizon:
        - if it is not settled, its terminal contribution is 0;
        - if a call is settled, it creates +1 share and -K cash at T_j,
          so its final contribution is x - K*exp(r*(T - T_j));
        - if a put is settled, it creates -1 share and +K cash at T_j,
          so its final contribution is K*exp(r*(T - T_j)) - x.

    For an option maturing at the final horizon, this reduces to the
    usual European payoff.
    """
    j = option["j"]
    K = option["K"]
    r = book["r"]
    T_final = book["T_final"]
    Tj = option["T"]

    if j == book["m"] - 1:
        # Final-maturity instruments are ordinary terminal payoffs.
        if option["kind"] == "call":
            return max(float(x) - K, 0.0)
        else:
            return max(K - float(x), 0.0)

    # Pre-final option. Exercise indicator is fixed by the pattern.
    A = math.exp(r * (T_final - Tj))
    component = pattern["components"][j]
    i = option["i"]

    if option["kind"] == "call":
        c = component["call"][i]
        return float(c) * (float(x) - K * A)
    else:
        p = component["put"][i]
        return float(p) * (K * A - float(x))


def option_tail_slope_coefficient(
    book: Dict[str, Any],
    option: Dict[str, Any],
    pattern: Dict[str, Any],
) -> float:
    """Right-tail slope coefficient of one net option position.

    The tail slope is the derivative of terminal wealth with respect to
    the final stock price x after the largest final-maturity strike.

    For pre-final physical settlements:
        settled call contributes +1 share -> slope +1;
        settled put  contributes -1 share -> slope -1.

    For final maturity:
        call payoff has tail slope +1;
        put payoff has tail slope 0.
    """
    j = option["j"]
    if j == book["m"] - 1:
        return 1.0 if option["kind"] == "call" else 0.0

    component = pattern["components"][j]
    i = option["i"]

    if option["kind"] == "call":
        return float(component["call"][i])
    else:
        return -float(component["put"][i])


# ------------------------------------------------------------
# 4. LP builder and solver
# ------------------------------------------------------------

def solve_multiperiod_arbitrage_detection(
    quotes: pd.DataFrame,
    S0: float,
    r: float,
    stock_bid: Optional[float] = None,
    stock_ask: Optional[float] = None,
    position_bound: float = 5.0,
    stock_position_bound: Optional[float] = None,
    tail_weight: float = 1.0,
    solver_options: Optional[Dict[str, Any]] = None,
) -> Tuple[Any, Dict[str, Any]]:
    """Solve the multiperiod deterministic arbitrage detection LP.

    Parameters
    ----------
    quotes:
        DataFrame with columns T, K, call_bid, call_ask, put_bid, put_ask.
    S0, r:
        Current stock price and continuously compounded risk-free rate.
    stock_bid, stock_ask:
        Optional executable stock quotes. If omitted, stock is frictionless.
    position_bound:
        Upper bound for each option buy/sell quantity. This is a numerical
        normalization/admissibility bound. Increasing it scales possible
        arbitrage certificates but can also make the LP less well conditioned.
    stock_position_bound:
        Upper bound for stock buy/sell quantities. Defaults to position_bound.
    tail_weight:
        Weight eta applied to right-tail slope margins in the objective.
    solver_options:
        Optional dictionary passed to scipy.optimize.linprog.

    Returns
    -------
    result, model:
        result is the SciPy linprog result.
        model stores the variable indexing and market data needed to inspect
        the solution.

    LP idea
    -------
    Maximize the sum of nonnegative finite-state margins and tail margins:

        sum_{pattern, x in X_final} D_{pattern,x}
        + tail_weight * sum_pattern M_pattern

    subject to:
        executable setup cost = 0,
        terminal wealth at every finite certificate point >= D,
        tail slope for every pattern >= M,
        D >= 0, M >= 0,
        buy/sell positions bounded and nonnegative.

    A strictly positive optimum means that at least one nonnegative payoff
    margin is strictly positive. This gives a deterministic arbitrage.
    """
    if stock_position_bound is None:
        stock_position_bound = position_bound

    book = build_market_book(quotes, S0=S0, r=r, stock_bid=stock_bid, stock_ask=stock_ask)
    patterns = enumerate_prefinal_patterns(book)
    x_points = final_certificate_points(book)

    if position_bound <= 0:
        raise ValueError("position_bound must be positive.")
    if stock_position_bound <= 0:
        raise ValueError("stock_position_bound must be positive.")

    # Variable creation helper.
    var_names: List[str] = []
    bounds: List[Tuple[Optional[float], Optional[float]]] = []

    def add_var(name, lb=0.0, ub=None):
        idx = len(var_names)
        var_names.append(name)
        bounds.append((lb, ub))
        return idx

    idx_cash = add_var("cash_b", lb=None, ub=None)

    # Stock is also split into buy/sell variables so that a stock spread
    # can be handled without changing the LP.
    idx_stock_buy = add_var("stock_buy_at_ask", lb=0.0, ub=stock_position_bound)
    idx_stock_sell = add_var("stock_sell_at_bid", lb=0.0, ub=stock_position_bound)

    # Option buy/sell variables.
    option_vars = []
    for k, opt in enumerate(book["option_records"]):
        buy_idx = add_var(f"BUY {opt['name']} at ask", lb=0.0, ub=position_bound)
        sell_idx = add_var(f"SELL {opt['name']} at bid", lb=0.0, ub=position_bound)
        rec = dict(opt)
        rec["buy_idx"] = buy_idx
        rec["sell_idx"] = sell_idx
        option_vars.append(rec)

    # Margin variables D_{pattern,x}.
    D_idx = {}
    for p_idx, _pat in enumerate(patterns):
        for x_idx, x in enumerate(x_points):
            D_idx[(p_idx, x_idx)] = add_var(
                f"D(pattern={p_idx}, x={x:g})", lb=0.0, ub=None
            )

    # Tail-slope margin variables M_pattern.
    M_idx = {}
    for p_idx, _pat in enumerate(patterns):
        M_idx[p_idx] = add_var(f"M_tail(pattern={p_idx})", lb=0.0, ub=None)

    nvar = len(var_names)

    # Objective: scipy minimizes, so maximize by minimizing the negative.
    c = np.zeros(nvar)
    for idx in D_idx.values():
        c[idx] = -1.0
    for idx in M_idx.values():
        c[idx] = -float(tail_weight)

    # Equality constraint: executable time-zero cost is exactly zero.
    cost_row = np.zeros(nvar)
    cost_row[idx_cash] = 1.0
    cost_row[idx_stock_buy] = book["stock_ask"]
    cost_row[idx_stock_sell] = -book["stock_bid"]

    for opt in option_vars:
        cost_row[opt["buy_idx"]] = opt["ask"]
        cost_row[opt["sell_idx"]] = -opt["bid"]

    A_eq = [cost_row]
    b_eq = [0.0]

    A_ub = []
    b_ub = []

    exp_rT = math.exp(book["r"] * book["T_final"])

    def terminal_row(pattern, x):
        """Linear coefficients of terminal wealth G^pattern(x)."""
        row = np.zeros(nvar)
        row[idx_cash] = exp_rT
        row[idx_stock_buy] = float(x)
        row[idx_stock_sell] = -float(x)

        for opt in option_vars:
            coeff = option_terminal_coefficient(book, opt, pattern, x)
            row[opt["buy_idx"]] += coeff
            row[opt["sell_idx"]] -= coeff

        return row

    def tail_slope_row(pattern):
        """Linear coefficients of the right-tail slope L^pattern."""
        row = np.zeros(nvar)
        row[idx_stock_buy] = 1.0
        row[idx_stock_sell] = -1.0

        for opt in option_vars:
            coeff = option_tail_slope_coefficient(book, opt, pattern)
            row[opt["buy_idx"]] += coeff
            row[opt["sell_idx"]] -= coeff

        return row

    # Nonnegativity at all finite certificate points:
    #     G(pattern, x) >= D(pattern, x)
    # becomes
    #    -G(pattern, x) + D(pattern, x) <= 0.
    for p_idx, pat in enumerate(patterns):
        for x_idx, x in enumerate(x_points):
            row = -terminal_row(pat, x)
            row[D_idx[(p_idx, x_idx)]] = 1.0
            A_ub.append(row)
            b_ub.append(0.0)

    # Nonnegative right-tail slope:
    #     L(pattern) >= M(pattern)
    # becomes
    #    -L(pattern) + M(pattern) <= 0.
    for p_idx, pat in enumerate(patterns):
        row = -tail_slope_row(pat)
        row[M_idx[p_idx]] = 1.0
        A_ub.append(row)
        b_ub.append(0.0)

    if solver_options is None:
        solver_options = {}

    result = linprog(
        c=c,
        A_ub=np.array(A_ub),
        b_ub=np.array(b_ub),
        A_eq=np.array(A_eq),
        b_eq=np.array(b_eq),
        bounds=bounds,
        method="highs",
        options=solver_options,
    )

    model = {
        "book": book,
        "patterns": patterns,
        "x_points": x_points,
        "var_names": var_names,
        "bounds": bounds,
        "idx_cash": idx_cash,
        "idx_stock_buy": idx_stock_buy,
        "idx_stock_sell": idx_stock_sell,
        "option_vars": option_vars,
        "D_idx": D_idx,
        "M_idx": M_idx,
        "objective_vector": c,
        "A_eq": np.array(A_eq),
        "b_eq": np.array(b_eq),
        "A_ub": np.array(A_ub),
        "b_ub": np.array(b_ub),
        "terminal_row": terminal_row,
        "tail_slope_row": tail_slope_row,
    }

    return result, model


# ------------------------------------------------------------
# 5. Solution inspection helpers
# ------------------------------------------------------------

def detection_objective_value(result) -> float:
    """Return the maximized LP objective value."""
    if not result.success:
        return float("nan")
    return -float(result.fun)


def extract_portfolio(result, model: Dict[str, Any], zero_tol: float = 1e-9):
    """Extract readable portfolio positions from a successful LP result."""
    if not result.success:
        raise RuntimeError(f"LP did not solve successfully: {result.message}")

    x = result.x
    book = model["book"]

    cash_b = x[model["idx_cash"]]
    stock_buy = x[model["idx_stock_buy"]]
    stock_sell = x[model["idx_stock_sell"]]
    stock_net = stock_buy - stock_sell

    stock_table = pd.DataFrame([{
        "instrument": "stock",
        "buy_at_ask": stock_buy,
        "sell_at_bid": stock_sell,
        "net_position": stock_net,
        "bid": book["stock_bid"],
        "ask": book["stock_ask"],
        "cost_contribution": stock_buy * book["stock_ask"] - stock_sell * book["stock_bid"],
    }])

    rows = []
    for opt in model["option_vars"]:
        buy = x[opt["buy_idx"]]
        sell = x[opt["sell_idx"]]
        net = buy - sell
        if abs(net) > zero_tol or buy > zero_tol or sell > zero_tol:
            rows.append({
                "kind": opt["kind"],
                "T": opt["T"],
                "K": opt["K"],
                "instrument": opt["name"],
                "buy_at_ask": buy,
                "sell_at_bid": sell,
                "net_position": net,
                "bid": opt["bid"],
                "ask": opt["ask"],
                "cost_contribution": buy * opt["ask"] - sell * opt["bid"],
            })

    options_table = pd.DataFrame(rows)
    if options_table.empty:
        options_table = pd.DataFrame(columns=[
            "kind", "T", "K", "instrument",
            "buy_at_ask", "sell_at_bid", "net_position",
            "bid", "ask", "cost_contribution"
        ])

    executable_cost = (
        cash_b
        + stock_table["cost_contribution"].sum()
        + options_table["cost_contribution"].sum()
    )

    return {
        "cash_b": cash_b,
        "stock_table": stock_table,
        "options_table": options_table,
        "executable_cost": executable_cost,
        "objective_value": detection_objective_value(result),
    }


def terminal_wealth_from_solution(
    result,
    model: Dict[str, Any],
    path: List[float],
) -> float:
    """Evaluate terminal wealth G_theta(s) on an arbitrary path.

    The path must have one stock value per maturity:
        path[j] = S_{T_j}

    This function uses the explicit physical-settlement formula.
    """
    if not result.success:
        raise RuntimeError(f"LP did not solve successfully: {result.message}")

    book = model["book"]
    if len(path) != book["m"]:
        raise ValueError(f"path must have length {book['m']}.")

    z = result.x
    x_final = float(path[-1])
    T_final = book["T_final"]
    r = book["r"]

    cash_b = z[model["idx_cash"]]
    stock_net = z[model["idx_stock_buy"]] - z[model["idx_stock_sell"]]

    wealth = cash_b * math.exp(r * T_final) + stock_net * x_final

    for opt in model["option_vars"]:
        qty = z[opt["buy_idx"]] - z[opt["sell_idx"]]
        if abs(qty) < 1e-14:
            continue

        j = opt["j"]
        K = opt["K"]
        A = math.exp(r * (T_final - opt["T"]))

        if j == book["m"] - 1:
            if opt["kind"] == "call":
                payoff = max(x_final - K, 0.0)
            else:
                payoff = max(K - x_final, 0.0)
        else:
            s_j = float(path[j])
            if opt["kind"] == "call":
                payoff = (x_final - K * A) if (s_j > K) else 0.0
            else:
                payoff = (K * A - x_final) if (s_j < K) else 0.0

        wealth += qty * payoff

    return float(wealth)


def certificate_table(result, model: Dict[str, Any]) -> pd.DataFrame:
    """Return the finite nonnegativity certificate values."""
    if not result.success:
        raise RuntimeError(f"LP did not solve successfully: {result.message}")

    z = result.x
    rows = []
    for p_idx, pat in enumerate(model["patterns"]):
        for x_idx, xval in enumerate(model["x_points"]):
            G = float(model["terminal_row"](pat, xval) @ z)
            D = z[model["D_idx"][(p_idx, x_idx)]]
            rows.append({
                "pattern_index": p_idx,
                "final_x": xval,
                "terminal_wealth_G": G,
                "margin_D": D,
                "pattern": pat["label"],
            })
        L = float(model["tail_slope_row"](pat) @ z)
        M = z[model["M_idx"][p_idx]]
        rows.append({
            "pattern_index": p_idx,
            "final_x": np.inf,
            "terminal_wealth_G": np.nan,
            "margin_D": np.nan,
            "tail_slope_L": L,
            "tail_margin_M": M,
            "pattern": pat["label"],
        })

    return pd.DataFrame(rows)


def print_arbitrage_report(
    result,
    model: Dict[str, Any],
    arbitrage_tol: float = 1e-7,
    show_certificate_rows: int = 12,
):
    """Print a compact but informative report for the detector result."""
    print("Solver status:", result.message)
    if not result.success:
        return

    W = detection_objective_value(result)
    print(f"Maximized objective value: {W:.12g}")

    if W > arbitrage_tol:
        print("Conclusion: deterministic multiperiod arbitrage FOUND.")
    else:
        print("Conclusion: no bounded arbitrage certificate found at this tolerance.")

    portfolio = extract_portfolio(result, model)
    print(f"\nZero-cost check, executable setup cost: {portfolio['executable_cost']:.12g}")
    print(f"Initial cash position b: {portfolio['cash_b']:.12g}")

    print("\nStock position:")
    display(portfolio["stock_table"])

    print("\nNonzero option positions:")
    display(portfolio["options_table"])

    cert = certificate_table(result, model)
    print("\nFirst rows of finite certificate table:")
    display(cert.head(show_certificate_rows))

    print("\nMinimum finite terminal-wealth certificate:",
          np.nanmin(cert["terminal_wealth_G"].to_numpy(float)))
    if "tail_slope_L" in cert.columns:
        print("Minimum patternwise tail slope:",
              np.nanmin(cert["tail_slope_L"].to_numpy(float)))


# ------------------------------------------------------------
# 6. Small demo data
# ------------------------------------------------------------

def demo_calendar_call_violation_quotes() -> pd.DataFrame:
    """Create a small two-maturity quote table with a call-calendar violation.

    With r >= 0 and physical settlement/passive carry, a longer maturity
    call with the same strike cannot be cheaper than a shorter maturity
    call in the executable bid-ask sense. In particular, the strategy

        buy longer call at ask,
        sell shorter call at bid

    should not have negative initial cost. The demo deliberately violates
    this by setting:

        C_bid(T1, K=100) > C_ask(T2, K=100).
    """
    return pd.DataFrame({
        "T":        [0.5,   1.0],
        "K":        [100.0, 100.0],
        "call_bid": [11.9,  10.9],
        "call_ask": [12.1,  11.1],
        # Puts are deliberately wide here. The example is meant to
        # isolate the multi-maturity call-calendar inconsistency rather
        # than create a tight put-specific demonstration.
        "put_bid":  [8.0,   5.0],
        "put_ask":  [15.0,  15.0],
    })



## Cell 3 - Demo: executable calendar violation

This small example contains two maturities and one strike. It deliberately violates the executable call-calendar condition:

\[
C^{bid}(T_1,K) > C^{ask}(T_2,K).
\]

The LP may return any normalized arbitrage portfolio satisfying the paper's conditions; it is not forced to return the simplest hand-written calendar spread.


In [ ]:

# Demo parameters.
S0_demo = 100.0
r_demo = 0.02

quotes_demo = demo_calendar_call_violation_quotes()
display(quotes_demo)

# Solve the detector.
result, model = solve_multiperiod_arbitrage_detection(
    quotes=quotes_demo,
    S0=S0_demo,
    r=r_demo,
    position_bound=1.0,
    stock_position_bound=1.0,
    tail_weight=1.0,
)

print_arbitrage_report(result, model)

# Evaluate the returned terminal wealth on a few arbitrary paths.
# A path has one stock value per maturity: [S_T1, S_T2].
test_paths = [
    [50.0, 50.0],
    [50.0, 100.0],
    [150.0, 50.0],
    [150.0, 150.0],
    [100.0, 100.0],
]

print("\nTerminal wealth on selected paths:")
for path in test_paths:
    print(path, "->", terminal_wealth_from_solution(result, model, path))



## Cell 4 - Replace with your own data

Use this template for your own multi-maturity option book.

Required columns:

```text
T, K, call_bid, call_ask, put_bid, put_ask
```

Use `bid = ask = price` for frictionless/single-price data.


In [ ]:

# Example template. Replace this DataFrame with your own quotes or a CSV upload.

# from google.colab import files
# uploaded = files.upload()
# csv_name = next(iter(uploaded))
# my_quotes = pd.read_csv(csv_name)

my_quotes = pd.DataFrame({
    "T":        [0.25, 0.25, 0.50, 0.50],
    "K":        [95.0, 105.0, 95.0, 105.0],
    "call_bid": [8.0,  3.0,   8.5,  3.5],
    "call_ask": [8.4,  3.4,   8.9,  3.9],
    "put_bid":  [2.5,  7.0,   3.0,  7.5],
    "put_ask":  [2.9,  7.4,   3.4,  7.9],
})

S0 = 100.0
r = 0.02

result, model = solve_multiperiod_arbitrage_detection(
    quotes=my_quotes,
    S0=S0,
    r=r,
    position_bound=2.0,
    stock_position_bound=2.0,
    tail_weight=1.0,
)

print_arbitrage_report(result, model)
